# Explicabilidad

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer
from pinecone import Pinecone
import nltk
from nltk.corpus import stopwords
from typing import List, Dict, Any, int
from secret import pincone_api

nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [58]:
def get_shared_keywords(query: str, document: str, top_n: int = 5) -> List[str]:
    """
    Encuentra las palabras clave más relevantes compartidas entre una consulta y un documento.

    Args:
        query: Texto de la consulta del usuario
        document: Texto del documento a comparar
        top_n: Número máximo de palabras clave a devolver

    Returns:
        Lista de las top_n palabras clave compartidas, ordenadas por relevancia
    """
    # Vectorización TF-IDF
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform([query, document])

    # Obtener pesos y palabras
    query_weights, doc_weights = tfidf_matrix.toarray()
    vocabulary = vectorizer.get_feature_names_out()

    # Seleccionar palabras relevantes (presentes en ambos textos)
    relevant_words = [
        (word, query_weight + doc_weight)
        for word, query_weight, doc_weight in zip(vocabulary, query_weights, doc_weights)
        if query_weight > 0 and doc_weight > 0
    ]

    # Ordenar y seleccionar las mejores
    relevant_words.sort(key=lambda x: x[1], reverse=True)
    return [word.capitalize() for word, _ in relevant_words[:top_n]]


def format_search_result(result: Dict[str, Any], keywords: List[str], number_of_desition: int) -> str:
    """
    Formatea los resultados de búsqueda en un string legible.

    Args:
        result: Diccionario con los resultados de la búsqueda
        keywords: Lista de palabras clave relevantes

    Returns:
        String formateado con la información de la película
    """
    metadata = result['matches'][number_of_desition]['metadata']
    return (
        f"Título: {metadata['Series_Title']}\n"
        f"Resumen: {metadata['Overview']}\n"
        f"Palabras clave: {', '.join(keywords)}"
    )


def search_movies(index, query: str, n_results: int = 2) -> str:
    """
    Busca películas relevantes basadas en una consulta y devuelve el mejor resultado.

    Args:
        query: Consulta de búsqueda del usuario
        n_results: Número de resultados a considerar

    Returns:
        String formateado con la información de la película más relevante
    """
    # Realizar búsqueda semántica
    query_vector = model.encode(query).tolist()
    results = index.query(vector=query_vector, top_k=n_results, include_metadata=True)

    # Obtener palabras clave para el primer resultado
    first_result_text = results['matches'][0]['metadata']['text']
    keywords = get_shared_keywords(query, first_result_text)
    number_of_desition = 0

    # Si no hay palabras clave compartidas, intentar con el segundo resultado
    if not keywords and n_results > 1:
        second_result_text = results['matches'][1]['metadata']['text']
        keywords = get_shared_keywords(query, second_result_text)
        number_of_desition = 1

    return format_search_result(results, keywords, number_of_desition)

### Usando Conepine

In [ ]:
pc = Pinecone(api_key=pincone_api)
index_name = "movies"
dimension_embeddings = 384
index_2 = pc.Index(index_name,
                   dimension=dimension_embeddings)

In [38]:
# query = 'a history with elves and a ring'
query = 'jewelry, thieves, colors'
# query = 'ship, love, sinks'
# query = 'ogre who rescues a princess'
# query = 'A long time ago in a galaxy far, far away... May the Force be with you'
# query = 'To infinity... and beyond!'
# query = 'There\'s no place like home, think about a girl named Dorothy, a little dog named Toto, and a journey down a yellow brick road'
# query = 'Hakuna Matata, young lion cub in Africa who makes some very interesting friends after a difficult experience,  Pumbaa and a meerkat named Timon. They teach the young lion this famous phrase'

In [63]:
out_model = search_movies(index=index_2, query='a history with elves and a ring')
out_model

'Título: The Hobbit: The Desolation of Smaug\nResumen: The dwarves, along with Bilbo Baggins and Gandalf the Grey, continue their quest to reclaim Erebor, their homeland, from Smaug. Bilbo Baggins is in possession of a mysterious and magical ring.\nPalabras clave: Ring'

In [64]:
out_model = search_movies(index=index_2, query='jewelry, thieves, colors')
out_model

'Título: Reservoir Dogs\nResumen: When a simple jewelry heist goes horribly wrong, the surviving criminals begin to suspect that one of them is a police informant.\nPalabras clave: Jewelry'

In [65]:
out_model = search_movies(index=index_2, query='There\'s no place like home, think about a girl named Dorothy, a little dog named Toto, and a journey down a yellow brick road')
out_model

'Título: Coraline\nResumen: An adventurous 11-year-old girl finds another world that is a strangely idealized version of her frustrating home, but it has sinister secrets.\nPalabras clave: Girl, Home'